In [3]:
import os
import openmc

os.environ['OPENMC_CROSS_SECTIONS'] = os.path.expanduser(
    os.environ.get('OPENMC_CROSS_SECTIONS', '~/openmc_data/endf-b-vii.1-hdf5/cross_sections.xml')
)
openmc.config['cross_sections'] = os.environ['OPENMC_CROSS_SECTIONS']


print("开始构建深大微堆材料库...")

# =====================================================================
# 1. 定义核燃料：12.5% 富集度 UO2 陶瓷芯体
# =====================================================================
leu_fuel = openmc.Material(name='12.5% UO2 Fuel')
# 直接添加核素，U235 和 U238 加起来刚好等于 1.0 份的 U
leu_fuel.add_nuclide('U235', 0.1264) 
leu_fuel.add_nuclide('U238', 0.8736)
# 再加入 2.0 份的 O，完美构成 UO2 分子比例
leu_fuel.add_nuclide('O16', 2.0)
leu_fuel.set_density('g/cm3', 10.5)

# =====================================================================
# 2. 定义燃料包壳：锆合金 Zr-4
# =====================================================================
zircaloy = openmc.Material(name='Zr-4 Cladding')
zircaloy.add_nuclide('Zr90', 1.0)
zircaloy.set_density('g/cm3', 6.55)

# =====================================================================
# 3. 定义冷却剂与慢化剂：高纯去离子轻水
# =====================================================================
water = openmc.Material(name='Light Water')
water.add_nuclide('H1', 2.0)
water.add_nuclide('O16', 1.0)
water.set_density('g/cm3', 0.9982) # 常温冷态池水密度
water.add_s_alpha_beta('c_H_in_H2O') 

# =====================================================================
# 4. 定义反射层：金属铍 (Be)
# =====================================================================
beryllium = openmc.Material(name='Beryllium Reflector')
beryllium.add_nuclide('Be9', 1.0)
beryllium.set_density('g/cm3', 1.848)
beryllium.add_s_alpha_beta('c_Be_in_Be')

# =====================================================================
# 5. 定义中央控制棒：金属镉 (Cd) 吸收体
# =====================================================================
cadmium = openmc.Material(name='Cadmium Control Rod')
cadmium.add_nuclide('Cd113', 1.0)
cadmium.set_density('g/cm3', 8.65)

# =====================================================================
# 6. 打包并导出 XML
# =====================================================================
# 将所有建好的材料装入 Materials 集合大管家
materials = openmc.Materials([leu_fuel, zircaloy, water, beryllium, cadmium])

# 导出为输运内核能够读取的底层文件
materials.export_to_xml()

print("✅ 所有关键材料已成功硬编码，并成功导出 materials.xml 文件！")




开始构建深大微堆材料库...
✅ 所有关键材料已成功硬编码，并成功导出 materials.xml 文件！


In [ ]:
import math
import openmc

print("开始进行深大微堆 355 个精确孔位的高保真空间拓扑计算...")

# =====================================================================
# 1. 严格对齐深大微堆工程图纸的尺寸参数 (单位: cm)
# =====================================================================
pitch = 1.095               # 栅距 (10.95 mm)

# --- 微观栅元尺寸 ---
# 1. 燃料棒 (总外径 4.3 mm)
r_fuel_meat = 0.215         # 燃料芯体半径
r_fuel_clad = 0.275         # 燃料包壳外半径

# 2. 控制棒 (芯体 3.9 mm, 不锈钢包壳厚 0.5 mm)
r_cr_meat = 0.195           # 控制棒芯体半径
r_cr_clad = 0.245           # 控制棒包壳外半径

# 3. 铝合金连接杆 (假设外径与燃料棒一致为 4.3 mm)
r_conn = 0.215              

# --- 宏观套娃尺寸 ---
r_core = 11.5               # 堆芯“鸟笼”外半径 (直径 230 mm / 2)
r_be_in = 11.55             # 铍层内侧半径 (直径 231 mm / 2)
r_be_out = 21.75            # 铍层外侧半径 (直径 435 mm / 2)
r_vessel_in = 30.0          # 反应堆容器内半径 (直径 600 mm / 2)
r_vessel_out = 31.0         # 反应堆容器外半径 (直径 620 mm / 2)

# 各层精确孔位数 (11层，总计 355)
ring_distribution = [1, 6, 12, 19, 26, 32, 39, 45, 52, 58, 65]

# =====================================================================
# 2. 初始化宏观区域
# =====================================================================
core_cells = []
core_boundary = openmc.ZCylinder(r=r_core)
water_region = -core_boundary

pin_count = 0

# =====================================================================
# 3. 高精度空间封装 (分别赋予不同的几何尺寸)
# =====================================================================
for ring_idx, num_pins in enumerate(ring_distribution):
    R = ring_idx * pitch  

    for i in range(num_pins):
        theta = 0.0 if num_pins == 1 else (2 * math.pi * i) / num_pins
        x = R * math.cos(theta)
        y = R * math.sin(theta)

        # -------------------------------------------------------------
        # 判断并装填不同尺寸的棒
        # -------------------------------------------------------------
        if ring_idx == 0:
            # 【第 0 圈：中央控制棒】(专属 4.9 mm 外径面)
            meat_cyl = openmc.ZCylinder(x0=x, y0=y, r=r_cr_meat)
            clad_cyl = openmc.ZCylinder(x0=x, y0=y, r=r_cr_clad)
            
            meat_cell = openmc.Cell(fill=cadmium, region=-meat_cyl)
            clad_cell = openmc.Cell(fill=zircaloy, region=+meat_cyl & -clad_cyl)
            
            core_cells.extend([meat_cell, clad_cell])
            water_region &= +clad_cyl

        elif ring_idx == 8 and (i % 13 == 0):
            # 【第 8 圈：四个均匀分布的连接杆】(实心铝棒)
            rod_cyl = openmc.ZCylinder(x0=x, y0=y, r=r_conn)
            rod_cell = openmc.Cell(fill=zircaloy, region=-rod_cyl) 
            
            core_cells.append(rod_cell)
            water_region &= +rod_cyl

        else:
            # 【其余 350 个孔位：燃料棒】(专属 4.3 mm 外径面)
            meat_cyl = openmc.ZCylinder(x0=x, y0=y, r=r_fuel_meat)
            clad_cyl = openmc.ZCylinder(x0=x, y0=y, r=r_fuel_clad)
            
            # 使用 leu_fuel 代表你的低浓化改造设计
            meat_cell = openmc.Cell(fill=leu_fuel, region=-meat_cyl)
            clad_cell = openmc.Cell(fill=zircaloy, region=+meat_cyl & -clad_cyl)
            
            core_cells.extend([meat_cell, clad_cell])
            water_region &= +clad_cyl
            
        pin_count += 1

# =====================================================================
# 4. 组装堆芯、水隙、铍反射层与反应堆容器
# =====================================================================
water_cell = openmc.Cell(fill=water, region=water_region)
core_cells.append(water_cell)

be_inner_boundary = openmc.ZCylinder(r=r_be_in)
water_gap_cell = openmc.Cell(fill=water, region=+core_boundary & -be_inner_boundary)
core_cells.append(water_gap_cell)

be_outer_boundary = openmc.ZCylinder(r=r_be_out)
be_reflector_cell = openmc.Cell(fill=beryllium, region=+be_inner_boundary & -be_outer_boundary)
core_cells.append(be_reflector_cell)

vessel_inner_boundary = openmc.ZCylinder(r=r_vessel_in)
pool_water_cell = openmc.Cell(fill=water, region=+be_outer_boundary & -vessel_inner_boundary)
core_cells.append(pool_water_cell)

vessel_outer_boundary = openmc.ZCylinder(r=r_vessel_out, boundary_type='vacuum')
vessel_wall_cell = openmc.Cell(fill=zircaloy, region=+vessel_inner_boundary & -vessel_outer_boundary)
core_cells.append(vessel_wall_cell)

# 导出
root_universe = openmc.Universe(cells=core_cells)
geometry = openmc.Geometry(root_universe)
geometry.export_to_xml()

print("✅ 全堆芯高保真 3D 几何（区分燃料棒与控制棒异质尺寸）拓扑构建完成！")

开始进行深大微堆 355 个精确孔位的高保真空间拓扑计算...
✅ 全堆芯高保真 3D 几何（区分燃料棒与控制棒异质尺寸）拓扑构建完成！
